In [3]:
# use langchain and openai key to generate recommendations based on SHAP impactful features
# import necessary libraries
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:

# Initialize model (GPT-4o-mini is fast and cheap)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Generate a completion
response = llm.invoke("Write a short limerick about football strategy. Keep it under 100 words.")
print(response.content)


In a huddle, the coach lays the plan,  
With a playbook as thick as a fan.  
"Spread wide, then cut in,  
Let the game now begin,  
And we'll score with a well-timed slam!"


In [10]:
# parent directory
os.path.abspath(os.path.join(os.getcwd(), '..'))

'/mnt/d/Projects/AIcoach/src'

In [11]:
# add src to the path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [15]:
from prompts import prompt1, prompt_opponent

In [16]:
# read json files for home and away impactful features
import json
with open('./player_improvement_suggestions_home.json', 'r') as f:
    home_features = json.load(f)
with open('./player_improvement_suggestions_away.json', 'r') as f:
    away_features = json.load(f)

In [30]:
home_features[0]['match_date'].split('-')[0]

'2025'

In [31]:
input_data_format = []
for idx, i in enumerate(home_features):
    # format home players
    player_data = ""
    # skip if match in 2026
    if i['match_date'].split('-')[0] == '2026':
        continue
    for k,v in i['home_players'].items():
        player_data += f"Player Name: {k}\n"
        for stat, value in v.items():
            if stat != 'position':
                # round value to 3 decimal places
                value = round(float(value), 4)
            player_data += f"  {stat}: {value}\n"

    opponent_data = ""
    for k,v in away_features[idx]['away_players'].items():
        opponent_data += f"Player Name: {k}\n"
        for stat, value in v.items():
            if stat != 'position':
                # round value to 3 decimal places
                value = round(float(value), 4)
            opponent_data += f"  {stat}: {value}\n"

    input_data_format.append({"player_data": player_data, "team": i['home'] , "opponent_team": i["away"], "date": i["match_date"], "opponent_data": opponent_data})

In [35]:
 away_features[0]['home_players']

{'Phil Foden': {'position': 'CM',
  'Gls': -0.23264188900589944,
  'Def Pen': -0.009504968523979085,
  'Crs': -0.00445902705192569,
  'GCA': -0.0029411223530769748,
  'PPA': -0.0028779524564742642,
  'PrgDist': -0.0024076828360557467,
  'Mis': -0.0015228298306465549},
 'Erling Haaland': {'position': 'FW',
  'PrgDist': -0.12080592006444935,
  'TotDist': -0.01770718604326238,
  'Crs': -0.015675976872444042,
  'TklW': -0.00666282534599294,
  'Gls': -0.004588264822959931,
  'familiarity': -0.003547988533973756,
  'Tkl+Int': -0.0026135353744030088},
 'Tijjani Reijnders': {'position': 'CM',
  'Gls': -0.04925480090081702,
  'Mis': -0.015018307566642686,
  'TklW': -0.007752684354782158,
  'Tkl+Int': -0.007570943534374264,
  'GCA': -0.004980521202087362,
  'Tkl%': -0.004046921506524026,
  'xcord': -0.0025332933664321544}}

In [36]:
for idx, i in enumerate(away_features):
    # format home players
    player_data = ""
    # skip if match in 2026
    if i['match_date'].split('-')[0] == '2026':
        continue
    for k,v in i['away_players'].items():
        player_data += f"Player Name: {k}\n"
        for stat, value in v.items():
            if stat != 'position':
                # round value to 3 decimal places
                value = round(float(value), 4)
            player_data += f"  {stat}: {value}\n"

    opponent_data = ""
    for k,v in away_features[idx]['home_players'].items():
        opponent_data += f"Player Name: {k}\n"
        for stat, value in v.items():
            if stat != 'position':
                # round value to 3 decimal places
                value = round(float(value), 4)
            opponent_data += f"  {stat}: {value}\n"
    input_data_format.append({"player_data": player_data, "team": i['away'] , "opponent_team": i["home"], "date": i["match_date"], "opponent_data": opponent_data})

In [37]:

# Initialize model (GPT-4o-mini is fast and cheap)
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

In [42]:
record_match_responses = []
for i in input_data_format:
    # use prompt1 from player_data
    final_prompt = prompt1.format(player_data=i['player_data'])
    final_opp_prompt = prompt_opponent.format(opponent_data=i['opponent_data'])

    # create llm chain
    llm_chain = LLMChain(llm=llm, prompt=prompt1)
    llm_chain_opp = LLMChain(llm=llm, prompt=prompt_opponent)
    # get response
    response = llm_chain.run(player_data=i['player_data'])
    response_opp = llm_chain_opp.run(opponent_data=i['opponent_data'])

    record_match_responses.append({"team": i['team'], "opponent_team": i['opponent_team'], "date": i['date'], "player_recommendations": response, "opponent_recommendations": response_opp})

In [43]:
import pandas as pd
pd.DataFrame(record_match_responses).to_csv('./shantam_llm_generation_recommendations.csv', index=False)